# Import important libraries

In [1]:
import numpy as np
from pathlib import Path
import torch
import glob
from PIL import Image
import pandas as pd
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader, Subset
import torchvision.transforms as transforms
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torchvision import models
from torchvision.models import ResNet18_Weights
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler
import skimage.io as skio
from skimage.feature import graycomatrix, graycoprops
import skimage.measure as skm
from skimage.filters import threshold_otsu 
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# unzip dataset
'''
!unzip -q "/content/drive/MyDrive/BMET5933/AS2/test-20260521T163436Z-3-001.zip" -d "/content/drive/MyDrive/BMET5933/AS2"
!unzip -q "/content/drive/MyDrive/BMET5933/AS2/train-20260521T163436Z-3-001.zip" -d "/content/drive/MyDrive/BMET5933/AS2"
!unzip -q "/content/drive/MyDrive/BMET5933/AS2/val-20260521T163437Z-3-001.zip" -d "/content/drive/MyDrive/BMET5933/AS2"
'''

In [5]:
# Hyperparameters configuration
image_size = 224
batch_size = 32
num_classes = 4
learning_rate = 1e-4
num_epochs = 30

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Dataset class definition

In [ ]:
# Dataset loading and preprocessing
class HandcraftedFeatures:
	'''
	input: PIL image
	output: list of handcrafted features
	'''
	# Convert the image to grayscale
	@staticmethod
	def get_gray_image(image):
		if image.mode != 'L':
			return np.array(image.convert('L'))
		return np.array(image)
	
	# Otsu's thresholding to convert the image to binary
	@staticmethod
	def get_binary_mask(gray_image):
		thresh = threshold_otsu(gray_image)
		binary_mask = gray_image > thresh
		return binary_mask

	@staticmethod
	def extract_shape_features(image):
		# Example feature extraction using skimage
		gray_image = HandcraftedFeatures.get_gray_image(image)	
		# Otsu's thresholding to convert the image to binary
		binary_mask = HandcraftedFeatures.get_binary_mask(gray_image)
		# Label connected components in the binary mask
		labeled_mask = skm.label(binary_mask)
		regions = skm.regionprops(labeled_mask)
		# Extract features from the largest connected component
		largest_region = max(regions, key=lambda r: r.area)
		area = largest_region.area
		perimeter = largest_region.perimeter
		solidity = largest_region.solidity
		eccentricity = largest_region.eccentricity
		return [area, perimeter, solidity, eccentricity]
	
	@staticmethod
	def extract_texture_features(image):
		# Example feature extraction using skimage
		gray_image = HandcraftedFeatures.get_gray_image(image)
		# Otsu's thresholding to convert the image to binary
		binary_mask = HandcraftedFeatures.get_binary_mask(gray_image)
		# Label connected components in the binary mask
		labeled_mask = skm.label(binary_mask)
		regions = skm.regionprops(labeled_mask)
		# Extract features from the largest connected component
		largest_region = max(regions, key=lambda r: r.area)
		masked_image = gray_image * (labeled_mask == largest_region.label)
		texture_features = []
		angles = [0, np.pi/4, np.pi/2, 3*np.pi/4]
		glcm_angle = graycomatrix(np.array(masked_image), distances=[1], angles=angles, levels=256, symmetric=True, normed=True)
		for i, _ in enumerate(angles):
			contrast = graycoprops(glcm_angle, 'contrast')[0, i]
			dissimilarity = graycoprops(glcm_angle, 'dissimilarity')[0, i]
			homogeneity = graycoprops(glcm_angle, 'homogeneity')[0, i]
			texture_features.extend([contrast, dissimilarity, homogeneity])
		return texture_features
	
	@staticmethod
	def extract_color_features(image):
		# feature extraction using skimage
		rgb_image = np.array(image)
		mean_red = np.mean(rgb_image[:, :, 0])
		mean_green = np.mean(rgb_image[:, :, 1])
		mean_blue = np.mean(rgb_image[:, :, 2])
		return [mean_red, mean_green, mean_blue]
	
	# Combine all features into a single feature vector
	@staticmethod
	def extract_all(image):
		shape_features = HandcraftedFeatures.extract_shape_features(image)
		texture_features = HandcraftedFeatures.extract_texture_features(image)
		color_features = HandcraftedFeatures.extract_color_features(image)
		return shape_features + texture_features + color_features

# Encode labels
label_encoder = LabelEncoder()
labels = [path.parent.name for path in Path("/content/drive/MyDrive/kidney_dataset/train/train").glob("*/*.jpg")]
label_encoder.fit(labels)

# Define the dataset class
class Kidney_Dataset(Dataset):
	def __init__(self, dataset_root, transform=None, label_encoder=None):
		super().__init__()
		self.dataset_root = dataset_root
		self.transform = transform
		self.image_paths = list(self.dataset_root.glob("*/*.jpg"))
		self.labels = [path.parent.name for path in self.image_paths]
		self.label_encoder = label_encoder
	
	def encode_label(self, label):
		# label encoder need and output list
		return self.label_encoder.transform([label])[0]
	
	def decode_label(self, encoded_label):
		return self.label_encoder.inverse_transform([encoded_label])[0]
	
	def __len__(self):
		return len(self.image_paths)
	
	def __getitem__(self, idx):
		image_path = self.image_paths[idx]
		label = self.labels[idx]
		
		original_image = Image.open(image_path).convert("RGB")

		# Extract handcrafted features
		handcrafted_features = torch.tensor(HandcraftedFeatures.extract_all(original_image), dtype=torch.float32)

		if self.transform:
			transformed_image = self.transform(original_image)
		
		encoded_label = self.encode_label(label)
		return transformed_image, handcrafted_features, encoded_label
	
	
# Define transformations for the dataset
transform = transforms.Compose(
	[transforms.Resize((image_size, image_size)),
	 transforms.ToTensor(),
	 transforms.Normalize(
		 mean = [0.485, 0.456, 0.406],
		 std = [0.229, 0.224, 0.225]
	 )]
)


# Dataset initialization

In [ ]:
train_root = Path("/content/drive/MyDrive/kidney_dataset/train")
validation_root = Path("/content/drive/MyDrive/kidney_dataset/validation")
test_root = Path("/content/drive/MyDrive/kidney_dataset/test")

train_dataset = Kidney_Dataset(train_root, transform=transform, label_encoder=label_encoder)
validation_dataset = Kidney_Dataset(validation_root, transform=transform, label_encoder=label_encoder)
test_dataset = Kidney_Dataset(test_root, transform=transform, label_encoder=label_encoder)

In [ ]:
# Dataloaders for training, validation and testing
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
validation_loader = DataLoader(validation_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

# Model custom modification

In [ ]:
class DL_model(nn.Module):
	def __init__(self):
		super().__init__()
		self.resnet = models.resnet18(weights = ResNet18_Weights.DEFAULT)
		self.feature_dim = self.resnet.fc.in_features
		self.resnet.fc = nn.Identity()  
	
	def forward(self, x):
		x = self.resnet(x)
		return x


class Fusion_model(nn.Module):
	'''
	fuse features from Resnet and handcrafted features
	'''
	def __init__(self, num_classes):
		super().__init__()
		self.resnet = DL_model()
		self.resnet_num_features = self.resnet.feature_dim
		self.handcrafted_num_features = 19
		
		# a simle mlp for fusion classification
		self.mlp = nn.Sequential(
			nn.Linear(self.resnet_num_features + self.handcrafted_num_features, 256),
			nn.ReLU(),
			nn.Dropout(0.3),
			nn.Linear(256, num_classes)
		)
	
	def forward(self, image, handcrafted_features):
		cnn_features = self.resnet(image)
		combined_features = torch.cat((cnn_features, handcrafted_features), dim=1)
		output = self.mlp(combined_features)
		return output

In [ ]:
# Initialize the model
model = Fusion_model(num_classes).to(device)
print(model)

# Define the loss function and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)
lr_scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs)

In [ ]:
def train_loop(model, train_loader, criterion, optimizer, lr_scheduler, device, num_epochs	):

	# Training loop
	for epoch in range(num_epochs):
		model.train()
		train_loss = 0.0
		correct = 0
		total = 0
		
		for images, handcrafted_features, labels in train_loader:
			images = images.to(device)
			handcrafted_features = handcrafted_features.to(device)
			labels = labels.to(device)
			
			# Forward pass and backward pass
			optimizer.zero_grad()
			outputs = model(images, handcrafted_features)
			loss = criterion(outputs, labels)
			loss.backward()
			optimizer.step()
			
			# Accumulate loss and calculate accuracy
			train_loss += loss.item() * images.size(0)
			predicted = torch.argmax(outputs, dim=1)
			correct += (predicted == labels).sum().item()
			total += labels.size(0)
		
		lr_scheduler.step()
		
		avg_train_loss = train_loss / len(train_loader.dataset)
		train_accuracy = correct / total
		print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {avg_train_loss:.4f}, Accuracy: {train_accuracy:.4f}")

# Evaluate the model on the validation set
def evaluate(model, validation_loader, criterion, device):
	model.eval()
	validation_loss = 0.0
	correct = 0
	total = 0
	# Do not calculate gradients during evaluation
	with torch.no_grad():
		for images, handcrafted_features, labels in validation_loader:
			images = images.to(device)
			handcrafted_features = handcrafted_features.to(device)
			labels = labels.to(device)
			
			outputs = model(images, handcrafted_features)
			loss = criterion(outputs, labels)
			
			validation_loss += loss.item() * images.size(0)
			predicted = torch.argmax(outputs, dim=1)
			correct += (predicted == labels).sum().item()
			total += labels.size(0)
	
	avg_validation_loss = validation_loss / len(validation_loader.dataset)
	validation_accuracy = correct / total
	print(f"Validation Loss: {avg_validation_loss:.4f}, Accuracy: {validation_accuracy:.4f}")
	return avg_validation_loss, validation_accuracy